<a href="https://colab.research.google.com/github/CemKeremSahin/NIR-to-RGB-Colorization/blob/main/Notebooks/U_NET_Train_3channels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
print(f"Sahip olduğun CPU çekirdek sayısı: {os.cpu_count()}")

In [ ]:
# --- 1. COLAB KURULUMLARI VE DRIVE BAĞLANTISI ---
import os

# Gerekli kütüphaneyi kuruyoruz
try:
    import segmentation_models_pytorch as smp
    print("Kütüphane zaten kurulu.")
except ImportError:
    print("Kütüphane kuruluyor...")
    os.system('pip install segmentation_models_pytorch')
    import segmentation_models_pytorch as smp

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Drive dan colab e aktarma
!cp -r /content/drive/MyDrive/4sensor_dataset/low_light_20230117/train /content/

In [ ]:
import os

# Görselinize göre güncellenmiş klasör yolları
DIR_785 = "/content/train/785nm"
DIR_850 = "/content/train/850nm"
DIR_940 = "/content/train/940nm"
RGB_FOLDER = "/content/train/gt_rgb"

print("--- 3 KANALLI (MULTI-BAND) VERİ SETİ KONTROLÜ BAŞLIYOR ---\n")

def veri_kontrol_3ch():
    klasorler = {
        "785nm": DIR_785,
        "850nm": DIR_850,
        "940nm": DIR_940,
        "GT_RGB": RGB_FOLDER
    }

    dosya_listeleri = {}

    # Tüm klasörlerin varlığını ve dosya sayılarını kontrol et
    for isim, yol in klasorler.items():
        if not os.path.exists(yol):
            print(f"HATA: {yol} klasörü bulunamadı! Lütfen yolu kontrol edin.")
            return

        # .bmp, .png veya .jpg uzantılı dosyaları okuyacak şekilde genişlettik
        dosyalar = sorted([f for f in os.listdir(yol) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        dosya_listeleri[isim] = dosyalar
        print(f"{isim} klasöründe {len(dosyalar)} adet görüntü dosyası bulundu.")

    # Tüm klasörlerdeki dosya sayılarının eşit olup olmadığını kontrol et
    uzunluklar = [len(liste) for liste in dosya_listeleri.values()]
    if len(set(uzunluklar)) == 1:
        print("\n>>> BAŞARILI: Tüm NIR bantları (785, 850, 940) ve RGB dosya sayıları EŞİT.")
    else:
        print("\n>>> UYARI: Dosya sayıları UYUŞMUYOR!")
        for isim, liste in dosya_listeleri.items():
            print(f"    {isim}: {len(liste)}")

    # Önceki kodunuzdaki 538 sayısı Real-World veri setinin toplamıdır[cite: 387].
    # Klasörünüzdeki sayı 458 çıkarsa bu normaldir çünkü 538 görüntünün 458'i eğitim (train), 80'i test içindir[cite: 388].

    # İlk 5 eşleşmeyi yan yana yazdırarak kontrol et
    print("\nÖrnek Eşleşmeler (İlk 5):")
    min_len = min(uzunluklar) if uzunluklar else 0

    if min_len > 0:
        for i in range(min(5, min_len)):
            f_785 = dosya_listeleri["785nm"][i]
            f_850 = dosya_listeleri["850nm"][i]
            f_940 = dosya_listeleri["940nm"][i]
            f_rgb = dosya_listeleri["GT_RGB"][i]
            print(f"  {f_785} | {f_850} | {f_940}  <--*-->  {f_rgb}")
    else:
        print("Klasörler boş olduğu için örnek gösterilemiyor.")

veri_kontrol_3ch()

--- 3 KANALLI (MULTI-BAND) VERİ SETİ KONTROLÜ BAŞLIYOR ---

785nm klasöründe 206 adet görüntü dosyası bulundu.
850nm klasöründe 206 adet görüntü dosyası bulundu.
940nm klasöründe 206 adet görüntü dosyası bulundu.
GT_RGB klasöründe 206 adet görüntü dosyası bulundu.

>>> BAŞARILI: Tüm NIR bantları (785, 850, 940) ve RGB dosya sayıları EŞİT.

Örnek Eşleşmeler (İlk 5):
  001.bmp | 001.bmp | 001.bmp  <--*-->  0001.bmp
  002.bmp | 002.bmp | 002.bmp  <--*-->  0002.bmp
  003.bmp | 003.bmp | 003.bmp  <--*-->  0003.bmp
  004.bmp | 004.bmp | 004.bmp  <--*-->  0004.bmp
  005.bmp | 005.bmp | 005.bmp  <--*-->  0005.bmp


In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from torchvision.transforms import Normalize
from tqdm import tqdm

# --- 1. (TF32 AKTİF) ---
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Kullanılan Cihaz: {DEVICE}")

# --- 2. AYARLAR VE YOLLAR ---
ROOT_PATH = "/content/train"
SAVE_PATH = "/content/drive/MyDrive/Computer_Vision/Model_Weights/U-NET_3Channels"
os.makedirs(SAVE_PATH, exist_ok=True)

PATCH_SIZE = 256
BATCH_SIZE = 16    # Makaledeki değer
NUM_EPOCHS = 3500  # Makaledeki değer
LEARNING_RATE = 2e-4 # Makaledeki değer
SAVE_EVERY_N_EPOCHS = 500

# --- 3. U-NET MİMARİSİ (GENERATOR) ---
class UNetDoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3): # in_channels 3 olarak güncellendi
        super().__init__()
        # Encoder
        self.inc = UNetDoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), UNetDoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), UNetDoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), UNetDoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), UNetDoubleConv(512, 1024))

        # Decoder & Skip Connections
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = UNetDoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = UNetDoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up3 = UNetDoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up4 = UNetDoubleConv(128, 64)

        self.outc = nn.Sequential(
            nn.Conv2d(64, out_channels, kernel_size=1),
            nn.Tanh() # Görüntü aralığını [-1, 1] yapar
        )

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.conv_up1(x)

        x = self.up2(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv_up2(x)

        x = self.up3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv_up3(x)

        x = self.up4(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv_up4(x)

        return self.outc(x)

# --- 4. VERİ SETİ (DATASET) ---
class MultiBandNIRDataset(Dataset):
    def __init__(self, root_path, patch_size=256):
        self.patch_size = patch_size
        self.data_cache = []

        # 3 Farklı NIR bandı ve RGB klasör yolları
        dir_785 = os.path.join(root_path, "785nm")
        dir_850 = os.path.join(root_path, "850nm")
        dir_940 = os.path.join(root_path, "940nm")
        rgb_dir = os.path.join(root_path, "gt_rgb")

        files_785 = sorted([f for f in os.listdir(dir_785) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        files_850 = sorted([f for f in os.listdir(dir_850) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        files_940 = sorted([f for f in os.listdir(dir_940) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        rgb_files = sorted([f for f in os.listdir(rgb_dir) if f.lower().endswith(('.bmp', '.png', '.jpg'))])

        print(f"📥 3 Kanallı Veriler RAM'e yükleniyor...")
        for f7, f8, f9, fr in tqdm(zip(files_785, files_850, files_940, rgb_files), total=len(files_785)):

            # Her bir bandı gri tonlamalı oku ve normalize et
            img_785 = cv2.imread(os.path.join(dir_785, f7), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
            img_850 = cv2.imread(os.path.join(dir_850, f8), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
            img_940 = cv2.imread(os.path.join(dir_940, f9), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0

            # 3 bandı kanal boyutunda (axis=-1) üst üste ekle -> Boyut: (H, W, 3)
            ir_img = np.stack([img_785, img_850, img_940], axis=-1)

            rgb_img = cv2.imread(os.path.join(rgb_dir, fr), cv2.IMREAD_COLOR)
            rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

            self.data_cache.append({'ir': ir_img, 'rgb': rgb_img})

    def __len__(self):
        return len(self.data_cache)

    def __getitem__(self, idx):
        d = self.data_cache[idx]

        # Numpy'dan PyTorch tensörüne çevir ve [-1, 1] aralığına çek
        ir = torch.from_numpy(d['ir']).permute(2, 0, 1) * 2.0 - 1.0
        rgb = torch.from_numpy(d['rgb']).permute(2, 0, 1) * 2.0 - 1.0

        # Rastgele kırpma (Random Crop)
        i, j, h, w = transforms.RandomCrop.get_params(ir, (self.patch_size, self.patch_size))
        return transforms.functional.crop(ir, i, j, h, w), transforms.functional.crop(rgb, i, j, h, w)

# --- 5. PERCEPTUAL LOSS (VGG NORMALİZASYONU EKLENDİ) ---
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.vgg = models.vgg16(weights='DEFAULT').features[:16].eval().to(DEVICE)
        for p in self.vgg.parameters():
            p.requires_grad = False

        self.criterion = nn.L1Loss()
        self.normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def forward(self, fake, real):
        norm_fake = self.normalize((fake + 1.0) / 2.0)
        norm_real = self.normalize((real + 1.0) / 2.0)
        return self.criterion(self.vgg(norm_fake), self.vgg(norm_real))

# --- 6. EĞİTİM KURULUMU VE DÖNGÜSÜ ---
# Modelin giriş kanalı (in_channels) 3 olarak başlatılıyor
model = UNetGenerator(in_channels=3, out_channels=3).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))

scaler = torch.amp.GradScaler('cuda')

criterion_pixel = nn.L1Loss()
criterion_vgg = PerceptualLoss()

# Güncellenmiş veri setini çağır
loader = DataLoader(MultiBandNIRDataset(ROOT_PATH, PATCH_SIZE), BATCH_SIZE, shuffle=True, pin_memory=True)

print(f"\n🚀 Klasik 3 Kanallı U-Net Eğitimi A100 Üzerinde Başlıyor (Standart Float16 AMP)...")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0
    loop = tqdm(loader, leave=False)

    for ir, real in loop:
        ir, real = ir.to(DEVICE), real.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda', dtype=torch.float16):
            output = model(ir)
            loss_pixel = criterion_pixel(output, real)
            loss_vgg = criterion_vgg(output, real)
            total_loss = loss_pixel + 0.1 * loss_vgg

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += total_loss.item()
        loop.set_description(f"Epoch [{epoch}/{NUM_EPOCHS}]")
        loop.set_postfix(Loss=total_loss.item())

    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Ortalama Kayıp: {epoch_loss/len(loader):.4f}")

    if epoch % SAVE_EVERY_N_EPOCHS == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
        }, os.path.join(SAVE_PATH, f"Classic_U-NET_Train_3channels_epoch_{epoch}.pth"))

print("✅ Eğitim tamamlandı. Drive senkronize ediliyor...")

from google.colab import drive
drive.flush_and_unmount()

print("✅ Senkronizasyon başarılı. Oturum kapatılıyor...")
from google.colab import runtime
runtime.unassign()